In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pandas_datareader import data as pdr
from datetime import datetime, timedelta

In [ ]:
class DataReader:
  def __init__(self, tickers, start, end, reader_params, l_cutoff=0.3, u_cutoff=0.7):
    self.start = start
    self.end = end
    self.reader_params = reader_params
    self.factors = self.read_factors(reader_params)

    self.nyse_pct = self.get_nyse_percentiles(l_cutoff, u_cutoff,end)
    self.market_data, self.BM_i, self.m_caps = self.read_market_data(tickers, start, end)


  def read_factors(self, reader_params):
    factors = pdr.DataReader(*self.reader_params)
    factor_data = factors[0]
    factor_data = factor_data[self.start:self.end]
    return factor_data

  def get_nyse_percentiles(self, l_cutoff, u_cutoff, end):
    nyse_tickers = [
        "NVDA", "AAPL", "GOOGL",
        "MSFT", "AMZN", "AVGO",
        "META", "TSLA", "BRK-B",
        "LLY", "WMT", "JPM",
        "V", "MA", "XOM", "ORCL",
        "UNH", "COST", "PG", "HD"
    ]

    bms = []

    for ticker in nyse_tickers:
      t = yf.Ticker(ticker)
      shares = t.info.get("sharesOutstanding")

      start = datetime.strptime(end, "%Y-%m-%d") - timedelta(days=1)
      history = t.history(start=start, end=end)['Close']

      market_cap = history.values[-1] * shares

      annual_bs = t.balance_sheet
      annual_bv = annual_bs.loc['Stockholders Equity'].iloc[::-1][-1]

      bms.append(annual_bv / market_cap)
    bms = np.array(bms)

    pl = np.quantile(bms, l_cutoff)
    ph = np.quantile(bms, u_cutoff)

    return [pl, ph]

  def read_market_data(self, tickers, start, end):
    price_data = {}
    market_caps = {}
    BMs = {}

    for ticker in tickers:
      t = yf.Ticker(ticker)
      shares = t.info.get("sharesOutstanding")
      history = t.history(start=start, end=end)['Close']

      market_cap = history.values[-1] * shares
      price_data[ticker] = history.pct_change().values

      annual_bs = t.balance_sheet
      annual_bv = annual_bs.loc['Stockholders Equity'].iloc[::-1][-1]

      market_caps[ticker] = market_cap
      BMs[ticker] = annual_bv / market_cap

    m_caps = pd.Series(market_caps)
    BM_i = pd.Series(BMs)

    m_data = pd.DataFrame(price_data)

    return m_data, BM_i, m_caps

  @property
  def datastore(self):
    return {
      "BM_i": self.BM_i,
      "nyse_percentiles": self.nyse_pct,
      "factors": self.factors,
      "market_data": self.market_data,
      "market_caps": self.m_caps
    }



In [ ]:
class DataStore(DataReader):
  def __init__(self, tickers, start, end, reader_params, l_cutoff=0.3, u_cutoff=0.7):
    super().__init__(tickers, start, end, reader_params)

    factors = self.datastore["factors"]
    market_data = self.datastore["market_data"]
    BM_i = self.datastore["BM_i"]
    self.m_caps = self.datastore["market_caps"]
    l_prct, u_prct = self.datastore["nyse_percentiles"]

    self.L, self.M, self.H = self.value_split(l_prct, u_prct, BM_i)
    self.S, self.B = self.sb_split()



  def value_split(self, l_prct, u_prct, BM_i):
    L = BM_i[BM_i < l_prct]
    M = BM_i[(BM_i > l_prct) & (BM_i < u_prct)]
    H = BM_i[BM_i > u_prct]

    return L, M, H

  def sb_split(self):
    m_cap_median = self.m_caps.median()

    S = self.m_caps[self.m_caps < m_cap_median]
    B = self.m_caps[self.m_caps > m_cap_median]

    return S, B

  def create_portfolios(self, market_data):
    md_cols = market_data.columns
    portfolios = {
      "SH_cols": md_cols[(md_cols in self.S.index) & (md_cols in self.H.index)],
      "BH_cols": md_cols[(md_cols in self.B.index) & (md_cols in self.H.index)],

      "SM_cols": md_cols[(md_cols in self.S.index) & (md_cols in self.M.index)],
      "BM_cols": md_cols[(md_cols in self.B.index) & (md_cols in self.M.index)],

      "SL_cols": md_cols[(md_cols in self.S.index) & (md_cols in self.L.index)],
      "BL_cols": md_cols[(md_cols in self.B.index) & (md_cols in self.L.index)]
    }

    for p in portfolios.keys():
      portfolios[p] = market_data[portfolios[p]]

    return portfolios


  def construct_factors(self):
    pass

  def _score(self):
    pass


In [ ]:
class FactorModel:
  def __init__(self, n_factors: int=5):
    self.featureset = None
    self.n_factors = n_factors
    self.model = None

  def fit(self):
    pass

In [ ]:
class FactorPremiaModel:
  def __init__(self):
    pass

  def fit(self):
    pass

In [ ]:
class HighRiskPortfolio():
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    self.model = FactorModel()
    self.premia_model = FactorModel()
    self.value_split()
    self.create_portfolios()
    self.construct_factors()
    self._init_model(self.model)
    self._init_factor_premia_model(self.premia_model)
    self.save_model = save_model
    self.model_path = model_path

  def _init_model(self):
    pass

  def _init_factor_premia_model(self):
    pass

  def _fit_factor_premia_model(self):
    pass
  
  def fit(self):
    if self.use_factor_premia_model:
      self._init_factor_premia_model()
      self._fit_factor_premia_model()

    self.model.fit()
    return self.model

  def _save_model(self):
    pass

  def build_portfolio(self):
    self._score()





In [ ]:
class PortfolioConstruction:
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    pass